## Configuration

### imports

In [1]:
## imports

# libraries
import os
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob

# custom
from Dense_Unet_Seg import *

print(tf.__version__)

2.4.0


In [2]:
# 设置用于训练的GPU
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
# The GPU id to use, usually either "0" or "1"
os.environ["CUDA_VISIBLE_DEVICES"]="1" #"1":train, "0":predict;

### Path Setting

In [3]:
#Linux
rootPath = '/home/shu/Dataset/Dataset-xVertSeg'
SrcmhdFolder = '/Spilt' #'/Resample'

## dataset_train, dataset_val构造

In [4]:
casestrainPath = rootPath + SrcmhdFolder + '/ROI/train/images/*/*.bin'
labelstrainPath = rootPath + SrcmhdFolder + '/ROI/train/masks/*/*.bin'

In [5]:
cases_train = sorted(glob(casestrainPath))
labels_train = sorted(glob(labelstrainPath))

In [6]:
len(labels_train)

150

In [7]:
dataset_train, count_train = dataset_train_build(casestrainPath,labelstrainPath)

[ 42  22  80  13  28   0   8  30   2  43  35  73 113 115  91 125  55  69
  23  66  61 112 134   4 133  65  74  72  12   6  81  97  34 132  58  68
 104  24 143  56 146 105  45  64 129  83 149 136  59  88 108  16 118  39
  92  29  67 127  77  46  48 107  37  96  99 144  60  47  26 142  98 122
 117  79 103   3  20  32 139  18  87  95  31 100  86  75 126  84 119 114
   9  51 131  27 135  50  89  17  78 137 111  11  53  76  40  15  90  82
 147  19 121 145  36  10  25 141  41 138 148  71  14 106  38 130   5 120
 140  52 123  54  49  63 124 110   1   7 101  33  70 102  93  21  62  44
  94 109 128  57  85 116]
['/home/shu/Dataset/Dataset-xVertSeg/Spilt/ROI/train/images/images_noisy/009.bin'
 '/home/shu/Dataset/Dataset-xVertSeg/Spilt/ROI/train/images/images_noisy/028.bin'
 '/home/shu/Dataset/Dataset-xVertSeg/Spilt/ROI/train/images/Splt_image006_etf/002.bin'
 '/home/shu/Dataset/Dataset-xVertSeg/Spilt/ROI/train/images/Splt_image009_etf/000.bin'
 '/home/shu/Dataset/Dataset-xVertSeg/Spilt/ROI/train

In [8]:
casesvalPath = rootPath + SrcmhdFolder + '/ROI/test/images/*/*.bin'
labelsvalPath = rootPath + SrcmhdFolder + '/ROI/test/masks/*/*.bin'

In [9]:
dataset_val, count_val = dataset_val_build(casesvalPath,labelsvalPath)

['/home/shu/Dataset/Dataset-xVertSeg/Spilt/ROI/test/images/Splt_image015/002.bin', '/home/shu/Dataset/Dataset-xVertSeg/Spilt/ROI/test/images/Splt_image015/003.bin', '/home/shu/Dataset/Dataset-xVertSeg/Spilt/ROI/test/images/Splt_image015/004.bin'] ['/home/shu/Dataset/Dataset-xVertSeg/Spilt/ROI/test/masks/Splt_image015/002.bin', '/home/shu/Dataset/Dataset-xVertSeg/Spilt/ROI/test/masks/Splt_image015/003.bin', '/home/shu/Dataset/Dataset-xVertSeg/Spilt/ROI/test/masks/Splt_image015/004.bin']


In [10]:
STEPS_PER_EPOCH = count_train//BATCH_SIZE
VALIDATION_STEPS = count_val//BATCH_SIZE

In [11]:
dataset_train,dataset_val

(<PrefetchDataset shapes: ((None, 80, 128, 112), (None, 80, 128, 112)), types: (tf.float32, tf.float64)>,
 <PrefetchDataset shapes: ((None, 80, 128, 112), (None, 80, 128, 112)), types: (tf.float32, tf.float64)>)

## 3D-Dense-U-Net训练

In [12]:
train(dataset_train,dataset_val,EPOCHS,STEPS_PER_EPOCH,VALIDATION_STEPS)

------------------------------
Creating and compiling model...
------------------------------
------------------------------
Fitting model...
------------------------------
Epoch 1/50
150/150 [==============================] - 156s 995ms/step - loss: 0.4961 - accuracy: 0.9176 - val_loss: 0.1969 - val_accuracy: 0.9445
Epoch 2/50
150/150 [==============================] - 157s 1s/step - loss: 0.1387 - accuracy: 0.9459 - val_loss: 0.1133 - val_accuracy: 0.9492
Epoch 3/50
150/150 [==============================] - 158s 1s/step - loss: 0.1083 - accuracy: 0.9537 - val_loss: 0.1062 - val_accuracy: 0.9577
Epoch 4/50
150/150 [==============================] - 159s 1s/step - loss: 0.0983 - accuracy: 0.9587 - val_loss: 0.1018 - val_accuracy: 0.9586
Epoch 5/50
150/150 [==============================] - 159s 1s/step - loss: 0.1010 - accuracy: 0.9588 - val_loss: 0.1005 - val_accuracy: 0.9595
Epoch 6/50
150/150 [==============================] - 160s 1s/step - loss: 0.1013 - accuracy: 0.9582 - val_lo

## 3D-Dense-U-Net预测

In [13]:
test_cases_folders = sorted(os.listdir(rootPath + SrcmhdFolder + '/ROI/test/images/'))#[0:1]
test_labels_folders = sorted(os.listdir(rootPath + SrcmhdFolder + '/ROI/test/masks/'))#[0:1]

In [14]:
# 依次加载测试数据集并预测
i = 0
for testfile in test_cases_folders:
    casesPath = rootPath + SrcmhdFolder + '/ROI/test/images/' + testfile + '/*.bin'
    labelsPath = rootPath + SrcmhdFolder + '/ROI/test/masks/' + test_labels_folders[i] + '/*.bin'
    # print(casesPath,labelsPath)
    dataset_test = dataset_test_build(casesPath,labelsPath)

    # 预测测试结果
    weight_dir = os.path.join('weights', '3D-Dense-Unet-Ins-20211001-112311' + '.h5')
    outPath = rootPath + SrcmhdFolder + "/pred_ROI/" + "pred_" + testfile + "/"
    predict(weight_dir,dataset_test,outPath,0.5)
    i = i + 1

['/home/shu/Dataset/Dataset-xVertSeg/Spilt/ROI/test/images/Splt_image011/002.bin', '/home/shu/Dataset/Dataset-xVertSeg/Spilt/ROI/test/images/Splt_image011/003.bin', '/home/shu/Dataset/Dataset-xVertSeg/Spilt/ROI/test/images/Splt_image011/004.bin'] ['/home/shu/Dataset/Dataset-xVertSeg/Spilt/ROI/test/masks/Splt_image011/002.bin', '/home/shu/Dataset/Dataset-xVertSeg/Spilt/ROI/test/masks/Splt_image011/003.bin', '/home/shu/Dataset/Dataset-xVertSeg/Spilt/ROI/test/masks/Splt_image011/004.bin']
------------------------------
Loading saved weights...
------------------------------
------------------------------
Predicting masks on test data...
------------------------------
(1, 80, 128, 112) (1, 80, 128, 112)
1/1 [==============================] - 1s 547ms/step
(1, 80, 128, 112) (1, 80, 128, 112)
1/1 [==============================] - 0s 260ms/step
(1, 80, 128, 112) (1, 80, 128, 112)
1/1 [==============================] - 0s 253ms/step
(1, 80, 128, 112) (1, 80, 128, 112)
1/1 [===================

## Eval Seg Results

In [15]:
# 定义评价指标:
EvalDict = {'DICE':0,'JACRD':0,'HDRFDST':0,'ACURCY':0}

In [16]:
GTROIPath = rootPath + SrcmhdFolder + '/ROI/test/masks/'
predROIPath = rootPath + SrcmhdFolder + '/pred_ROI/'

In [17]:
GTROIFolders = sorted(os.listdir(GTROIPath))#[4:5]
predROIFolders = sorted(os.listdir(predROIPath))#[4:5]

### 获得评价指标对应结果

In [20]:
i = 0
EvalResultsDatas = []
for GTROIFolder in GTROIFolders:
    GTFolder = GTROIPath + GTROIFolder
    PredFolder = predROIPath + predROIFolders[i] + '/rmvsml'
    
    EvalResultsData = EvalResults(GTFolder,PredFolder,EvalDict)
    EvalResultsData.index = ['L1', 'L2', 'L3', 'L4', 'L5']
    EvalResultsDatas.append(EvalResultsData)
    i+=1

5 5
       DICE     JACRD    HDRFDST    ACURCY
0  0.907470  0.830613   6.164414  0.992678
0  0.870210  0.770241  14.317821  0.989716
0  0.903662  0.824255   8.062258  0.991212
0  0.893446  0.807414   8.246211  0.988811
0  0.880706  0.786841   7.280110  0.988026
5 5
       DICE     JACRD    HDRFDST    ACURCY
0  0.884425  0.792797   8.306624  0.988783
0  0.888201  0.798886   7.000000  0.987938
0  0.888655  0.799622   8.602325  0.987895
0  0.880516  0.786537  11.832160  0.987248
0  0.848421  0.736745  13.747727  0.983505
5 5
       DICE     JACRD    HDRFDST    ACURCY
0  0.910997  0.836542   6.403124  0.990005
0  0.920001  0.851854   5.744563  0.990328
0  0.904850  0.826234   6.000000  0.987042
0  0.848552  0.736943  15.000000  0.983132
0  0.855727  0.747834  10.770330  0.979240
5 5
       DICE     JACRD    HDRFDST    ACURCY
0  0.771503  0.628006  14.764823  0.971206
0  0.809576  0.680074  15.652476  0.974749
0  0.917595  0.847737   8.831761  0.988166
0  0.914179  0.841924   9.643651  0.98

### 所有椎体的DICE平均值

In [108]:
ResultsALL = pd.concat(EvalResultsDatas)

In [109]:
format(ResultsALL.DICE.mean(), '3f')

'0.871446'

### Get Dice of L1-L5

In [110]:
# Get Mean Dice of L1-L5
L1 = []
L2 = []
L3 = []
L4 = []
L5 = []
for EvalResult in EvalResultsDatas:
    
    L1.append(EvalResult.DICE['L1'])
    L2.append(EvalResult.DICE['L2'])
    L3.append(EvalResult.DICE['L3'])
    L4.append(EvalResult.DICE['L4'])
    L5.append(EvalResult.DICE['L5'])

In [111]:
L = [L1,L2,L3,L4,L5]

In [112]:
MeanL = []
for L_idx in L:
    MeanL.append(format(np.array(L_idx).mean(), '.4f'))
print(MeanL)

['0.8628', '0.8705', '0.8902', '0.8734', '0.8603']


In [113]:
0.8720277999999999
['0.8634', '0.8713', '0.8904', '0.8738', '0.8613']

['0.8634', '0.8713', '0.8904', '0.8738', '0.8613']